# Fake News Detector - Colab Training

This notebook trains all model×embedding×dataset combinations on Google Colab, persisting every artifact to Google Drive so nothing is lost on session timeout.

## Workflow

1. Configuration
2. Mount Google Drive
3. Clone or update repository
4. Install dependencies
5. Create Drive folders
6. Upload raw datasets and embeddings
7. Validate and preprocess datasets
8. Configure project symlinks (Drive ↔ project)
9. Train models
10. Collect web data
11. Evaluate on web data
12. Analyze results

## 0 - Configuration

In [ ]:
from pathlib import Path

GITHUB_USERNAME = "wgrzesik"
REPO_NAME = "fake-news-detector"
BRANCH = "develop"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

DRIVE_ROOT = Path("/content/drive/MyDrive/fake-news-results")
PROJECT_DIR = Path(f"/content/{REPO_NAME}")

print("Bootstrap configuration ready")
print(f"Repository: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Project dir: {PROJECT_DIR}")

## 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 - Clone or update repository
First run clones; subsequent runs pull the latest changes. After pushing code changes locally, just re-run this cell.

In [ ]:
import os

if PROJECT_DIR.is_dir():
    print("Repo exists - pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git log --oneline -3

## 3 - Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4 - Create Drive folders
Create the required Google Drive folder structure for raw data, processed data, embeddings, and web data.

In [ ]:
RAW_REQUIREMENTS = {
    "ISOT": ["True.csv", "Fake.csv"],
    "LIAR": ["train.tsv", "test.tsv", "valid.tsv"],
    "WELFake": ["data.csv"],
}

REQUIRED_EMBEDDINGS = [
    DRIVE_ROOT / "datasets" / "embeddings" / "glove.6B.100d.txt",
]

required_dirs = [
    DRIVE_ROOT / "datasets" / "raw" / "ISOT",
    DRIVE_ROOT / "datasets" / "raw" / "LIAR",
    DRIVE_ROOT / "datasets" / "raw" / "WELFake",
    DRIVE_ROOT / "datasets" / "processed" / "ISOT",
    DRIVE_ROOT / "datasets" / "processed" / "LIAR",
    DRIVE_ROOT / "datasets" / "processed" / "WELFake",
    DRIVE_ROOT / "datasets" / "embeddings",
    DRIVE_ROOT / "datasets" / "web_scraped_data" / "processed",
]

for directory in required_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print("Drive folder structure created.")

## 5 - Upload raw datasets and embeddings
Upload files to Google Drive before preprocessing:

Raw datasets:
- ISOT: `/content/drive/MyDrive/fake-news-results/datasets/raw/ISOT/`:
`True.csv`, `Fake.csv`
- LIAR: `/content/drive/MyDrive/fake-news-results/datasets/raw/LIAR/`: `train.tsv`, `test.tsv`, `valid.tsv`
- WELFake: `/content/drive/MyDrive/fake-news-results/datasets/raw/WELFake/`: `data.csv`

Embedding file:
- GloVe: `/content/drive/MyDrive/fake-news-results/datasets/embeddings/glove.6B.100d.txt`

## 6 - Validate and preprocess datasets

In [ ]:
import os
import shutil
import subprocess
import sys

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

missing_raw = []
for dataset, filenames in RAW_REQUIREMENTS.items():
    raw_dir = DRIVE_ROOT / "datasets" / "raw" / dataset
    for filename in filenames:
        file_path = raw_dir / filename
        if not file_path.exists():
            missing_raw.append(file_path)

missing_embeddings = [path for path in REQUIRED_EMBEDDINGS if not path.exists()]

if missing_raw or missing_embeddings:
    if missing_raw:
        print("Missing required raw dataset files:")
        for path in missing_raw:
            print(f" - {path}")
    if missing_embeddings:
        print("\nMissing required embedding files:")
        for path in missing_embeddings:
            print(f" - {path}")
    raise FileNotFoundError("Upload missing raw datasets/embeddings to Drive and rerun this cell.")

print("All required raw datasets and embeddings found.")

for dataset in RAW_REQUIREMENTS:
    source = DRIVE_ROOT / "datasets" / "raw" / dataset
    target = PROJECT_DIR / "research" / dataset

    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)

    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)
    print(f"{target} -> {source}")

run([sys.executable, "-m", "research.preprocess_datasets"], cwd=PROJECT_DIR)

for dataset in RAW_REQUIREMENTS:
    processed_dir = DRIVE_ROOT / "datasets" / "processed" / dataset
    processed_dir.mkdir(parents=True, exist_ok=True)

    for split in ["train.csv", "test.csv", "val.csv"]:
        source_file = PROJECT_DIR / "research" / dataset / split
        if not source_file.exists():
            raise FileNotFoundError(f"Expected preprocessing output not found: {source_file}")
        destination = processed_dir / split
        shutil.copy2(source_file, destination)
        print(f"Copied: {source_file} -> {destination}")

print("Preprocessing stage complete.")

## 7 - Configure project symlinks (Drive ↔ project)
Links datasets/embeddings into the project and output dirs out to Drive.

In [ ]:
!python -m research.colab.colab_setup --project {PROJECT_DIR} --drive {DRIVE_ROOT}

## 8 - Train models
Each cell trains one dataset. Re-runnable: Optuna studies use load_if_exists=True, so interrupted runs resume where they left off.

Key Hydra overrides you can add:

- `datasets_list=[ISOT,LIAR]` — subset of datasets
- `models_to_optimize=[svm,xgb]` — subset of models
- `embeddings_to_use=[tfidf,glove]` — subset of embeddings
- `optuna.n_trials=20` — more tuning trials
- `hydra.job.chdir=Falsec — disables dynamic output folders to keep your Drive symlinks and relative paths intact

example:
```!python -m research.train_models \
    datasets_list=[LIAR] \
    models_to_optimize=[bert] \
    embeddings_to_use=[bert-base-uncased] \
    hydra.job.chdir=False

In [ ]:
!python -m research.train_models

## 9 - Collect web data
Scrapes real-world news articles, preprocesses with the same pipeline as training data.
Outputs three CSVs to `experiments/web_test_results/` (title, excerpt, full text) for
generalization testing on out-of-distribution data.

In [ ]:
!python -m research.web.collect_web

## 10 - Evaluate on web data
Runs all trained models against collected web data. Compares across datasets (ISOT, LIAR, WELFake),
model types (SVM, XGBoost, NN), embeddings (TF-IDF, GloVe, BERT), and text types (title, excerpt, full text).
Outputs predictions to `experiments/web_test_results/results_*.csv`.

In [ ]:
!python -m research.evaluate_web

## 11 - Analyze results
Aggregates all results into reports and visualizations. Computes F1/Precision/Recall across
all combinations, calculates generalization gaps, and generates charts saved to `presentation_charts/`.
Check `GENERALIZATION_SUMMARY.md` for a human-readable summary of top-performing models.

In [ ]:
!python -m research.analyze_results